1) SetUp para que todo funcione correctamente

In [133]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc

from typing import Optional
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report




PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.minio_client import download_df_parquet


print("✓ Setup completado")

✓ Setup completado


2) Extraemos desde MINIO los dos primeros meses de 2025

In [134]:
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")
if not access_key or not secret_key:
    raise ValueError("MINIO_ACCESS_KEY y MINIO_SECRET_KEY deben estar definidas")

print("Cargando datasets desde MinIO...")

MINIO_PATH = "grupo5/aggregations/DataFrameGroupedByMin=30.parquet"

df = download_df_parquet(access_key, secret_key, MINIO_PATH)

print(f"✓ Total:   {df.shape[0]:,} filas × {df.shape[1]} columnas")

Cargando datasets desde MinIO...
✓ Total:   21,324,492 filas × 82 columnas


In [135]:
TARGET_RAW = "alert_in_next_15m_max"
TARGET = "alert_in_next_15m_max"

def agregar_por_linea(df_raw):
    cols_base = [
        "route_id", "direction", "merge_time",
        "delay_seconds_mean",
        "actual_headway_seconds_mean",
        "is_unscheduled_max",
        "num_updates_sum",
        "match_key_nunique",
        "hour_sin_first",
        "hour_cos_first",
        "dow_first",
        "is_weekend_max",
        "seconds_since_last_alert_mean",
        TARGET_RAW,
    ]

    # Usamos solo una familia de lags
    if all(c in df_raw.columns for c in ["lagged_delay_1_mean", "lagged_delay_2_mean", "lagged_delay_3_mean"]):
        lag_cols = ["lagged_delay_1_mean", "lagged_delay_2_mean", "lagged_delay_3_mean"]
        lag_rename = {
            "lagged_delay_1_mean_mean": "delay_1_before_mean",
            "lagged_delay_2_mean_mean": "delay_2_before_mean",
            "lagged_delay_3_mean_mean": "delay_3_before_mean",
        }
    elif all(c in df_raw.columns for c in ["delay_1_before", "delay_2_before", "delay_3_before"]):
        lag_cols = ["delay_1_before", "delay_2_before", "delay_3_before"]
        lag_rename = {
            "delay_1_before_mean": "delay_1_before_mean",
            "delay_2_before_mean": "delay_2_before_mean",
            "delay_3_before_mean": "delay_3_before_mean",
        }
    else:
        lag_cols = []
        lag_rename = {}

    cols_usar = [c for c in cols_base + lag_cols if c in df_raw.columns]

    df = df_raw[cols_usar].copy()
    df = df[df[TARGET_RAW].notna()].copy()

    df["merge_time"] = pd.to_datetime(df["merge_time"])
    df[TARGET_RAW] = df[TARGET_RAW].astype(int)
    df["parada_retrasada"] = (df["delay_seconds_mean"] > 60).astype(int)

    agg_dict = {
        "delay_seconds_mean": ["mean", "max", "std"],
        "parada_retrasada": ["sum", "count"],
        "actual_headway_seconds_mean": ["mean", "std"],
        "is_unscheduled_max": "max",
        "num_updates_sum": "sum",
        "match_key_nunique": "sum",
        "hour_sin_first": "first",
        "hour_cos_first": "first",
        "dow_first": "first",
        "is_weekend_max": "max",
        "seconds_since_last_alert_mean": "min",
        TARGET_RAW: "max",
    }

    for c in lag_cols:
        agg_dict[c] = "mean"

    print("Agregando por línea...")
    df_linea = df.groupby(
        ["route_id", "direction", pd.Grouper(key="merge_time", freq="30min")],
        observed=True
    ).agg(agg_dict).reset_index()

    df_linea.columns = [
        "_".join(filter(None, col)) if isinstance(col, tuple) else col
        for col in df_linea.columns
    ]

    rename_dict = {
        "delay_seconds_mean_mean": "delay_mean_linea",
        "delay_seconds_mean_max": "delay_max_linea",
        "delay_seconds_mean_std": "delay_std_linea",
        "parada_retrasada_sum": "paradas_retrasadas",
        "parada_retrasada_count": "total_paradas",
        "actual_headway_seconds_mean_mean": "headway_mean_linea",
        "actual_headway_seconds_mean_std": "headway_std_linea",
        "is_unscheduled_max_max": "is_unscheduled",
        "num_updates_sum_sum": "num_updates",
        "match_key_nunique_sum": "match_key_nunique",
        "hour_sin_first_first": "hour_sin",
        "hour_cos_first_first": "hour_cos",
        "dow_first_first": "dow",
        "is_weekend_max_max": "is_weekend",
        "seconds_since_last_alert_mean_min": "seg_desde_ultima_alerta_linea",
        f"{TARGET_RAW}_max": TARGET,
        **lag_rename,
    }

    df_linea = df_linea.rename(columns=rename_dict)

    # seguridad por si hubiera columnas duplicadas
    df_linea = df_linea.loc[:, ~df_linea.columns.duplicated()].copy()

    # asegurar que existan las columnas lag
    for c in ["delay_1_before_mean", "delay_2_before_mean", "delay_3_before_mean"]:
        if c not in df_linea.columns:
            df_linea[c] = 0.0

    # imputación estructural a nivel agregado
    fill_zero_cols = [
        "delay_mean_linea",
        "delay_max_linea",
        "delay_std_linea",
        "delay_1_before_mean",
        "delay_2_before_mean",
        "delay_3_before_mean",
        "headway_mean_linea",
        "headway_std_linea",
        "is_unscheduled",
        "num_updates",
        "match_key_nunique",
        "hour_sin",
        "hour_cos",
        "dow",
        "is_weekend",
        "seg_desde_ultima_alerta_linea",
    ]

    for col in fill_zero_cols:
        if col in df_linea.columns:
            df_linea[col] = pd.to_numeric(df_linea[col], errors="coerce").fillna(0)

    df_linea["paradas_retrasadas"] = df_linea["paradas_retrasadas"].fillna(0)
    df_linea["total_paradas"] = df_linea["total_paradas"].fillna(0)

    df_linea["pct_paradas_retrasadas"] = (
        df_linea["paradas_retrasadas"] / df_linea["total_paradas"].clip(lower=1)
    )

    df_linea["delay_acceleration_linea"] = (
        df_linea["delay_mean_linea"] - df_linea["delay_1_before_mean"]
    )

    df_linea["headway_cv"] = (
        df_linea["headway_std_linea"] / df_linea["headway_mean_linea"].clip(lower=1)
    )

    df_linea["colapso_linea"] = (
        (df_linea["pct_paradas_retrasadas"] > 0.5).astype(int)
    )

    df_linea["delay_x_aceleracion"] = (
        df_linea["delay_mean_linea"] * df_linea["delay_acceleration_linea"].clip(lower=0)
    )

    df_linea["seg_desde_ultima_alerta_linea"] = (
        df_linea["seg_desde_ultima_alerta_linea"].fillna(999999)
    )

    df_linea = df_linea.dropna(subset=[TARGET]).copy()
    df_linea[TARGET] = df_linea[TARGET].astype(int)

    del df
    gc.collect()

    print("Columnas duplicadas:", df_linea.columns[df_linea.columns.duplicated()].tolist())
    print(f"Dataset línea: {df_linea.shape[0]:,} filas x {df_linea.shape[1]} columnas")
    print(f"Positivos: {df_linea[TARGET].mean()*100:.1f}%")

    return df_linea


def agregar_features_rolling_retraso(df_linea: pd.DataFrame) -> pd.DataFrame:
    df_linea = df_linea.sort_values(["route_id", "direction", "merge_time"]).copy()
    grp = df_linea.groupby(["route_id", "direction"])

    df_linea["delay_rolling4_mean"] = (
        grp["delay_mean_linea"]
        .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
        .fillna(0)
    )

    df_linea["delay_rolling4_max"] = (
        grp["delay_max_linea"]
        .transform(lambda x: x.shift(1).rolling(4, min_periods=1).max())
        .fillna(0)
    )

    df_linea["headway_rolling4_std"] = (
        grp["headway_mean_linea"]
        .transform(lambda x: x.shift(1).rolling(4, min_periods=1).std())
        .fillna(0)
    )

    return df_linea

In [136]:
df_linea = agregar_por_linea(df)
df_linea = agregar_features_rolling_retraso(df_linea)

print(df_linea.shape)
df_linea.head()

Agregando por línea...
Columnas duplicadas: []
Dataset línea: 755,443 filas x 27 columnas
Positivos: 14.4%
(755443, 30)


,route_id,direction,merge_time,delay_mean_linea,delay_max_linea,delay_std_linea,paradas_retrasadas,total_paradas,headway_mean_linea,headway_std_linea,...,delay_2_before_mean,delay_3_before_mean,pct_paradas_retrasadas,delay_acceleration_linea,headway_cv,colapso_linea,delay_x_aceleracion,delay_rolling4_mean,delay_rolling4_max,headway_rolling4_std
0,0,N,2025-02-26 02:30:00,0.0,0.0,0.0,0,15,620.400024,551.131149,...,0.0,0.0,0.0,0.0,0.888348,0,0.0,0.0,0.0,0.000000
1,0,N,2025-02-26 03:00:00,0.0,0.0,0.0,0,18,322.277771,309.536999,...,0.0,0.0,0.0,0.0,0.960466,0,0.0,0.0,0.0,0.000000
2,0,N,2025-02-26 03:30:00,0.0,0.0,0.0,0,8,265.250000,185.184657,...,0.0,0.0,0.0,0.0,0.698151,0,0.0,0.0,0.0,210.804267
3,0,N,2025-02-27 01:30:00,0.0,0.0,0.0,0,4,1130.750000,20.238165,...,0.0,0.0,0.0,0.0,0.017898,0,0.0,0.0,0.0,190.726936
4,0,N,2025-02-27 02:00:00,0.0,0.0,0.0,0,16,51.812500,35.799849,...,0.0,0.0,0.0,0.0,0.690950,0,0.0,0.0,0.0,395.962464


3) Preparamos el dataframe para trabajar con lo que necesita el modelo

In [137]:
features = [
    "delay_mean_linea",
    "delay_max_linea",
    "delay_std_linea",
    "paradas_retrasadas",
    "total_paradas",
    "pct_paradas_retrasadas",
    "delay_1_before_mean",
    "delay_2_before_mean",
    "delay_3_before_mean",
    "delay_acceleration_linea",
    "delay_x_aceleracion",
    "headway_mean_linea",
    "headway_std_linea",
    "headway_cv",
    "delay_rolling4_mean",
    "delay_rolling4_max",
    "headway_rolling4_std",
    "is_unscheduled",
    "num_updates",
    "match_key_nunique",
    "hour_sin",
    "hour_cos",
    "dow",
    "is_weekend",
    "seg_desde_ultima_alerta_linea",
    "colapso_linea",
    "direction",
    "route_id",
]

target = TARGET

cols_needed = features + [target, "merge_time"]

df_model = df_linea[cols_needed].copy()

print("Shape inicial:", df_model.shape)
print("\nNaN iniciales:")
print(df_model.isna().sum().sort_values(ascending=False))

Shape inicial: (755443, 30)

NaN iniciales:
delay_mean_linea                 0
delay_max_linea                  0
delay_std_linea                  0
paradas_retrasadas               0
total_paradas                    0
pct_paradas_retrasadas           0
delay_1_before_mean              0
delay_2_before_mean              0
delay_3_before_mean              0
delay_acceleration_linea         0
delay_x_aceleracion              0
headway_mean_linea               0
headway_std_linea                0
headway_cv                       0
delay_rolling4_mean              0
delay_rolling4_max               0
headway_rolling4_std             0
is_unscheduled                   0
num_updates                      0
match_key_nunique                0
hour_sin                         0
hour_cos                         0
dow                              0
is_weekend                       0
seg_desde_ultima_alerta_linea    0
colapso_linea                    0
direction                        0
route_id   

4) Limpieza inicial

In [138]:
df_model = df_model.dropna(subset=[target]).copy()
df_model[target] = pd.to_numeric(df_model[target], errors="coerce")
df_model = df_model.dropna(subset=[target]).copy()
df_model[target] = df_model[target].astype(int)

categorical_features = ["direction", "route_id"]

binary_features = [
    "is_unscheduled",
    "is_weekend",
    "colapso_linea",
]

numeric_features = [
    c for c in features if c not in categorical_features
]

# pasar inf a NaN
df_model[numeric_features] = df_model[numeric_features].replace([np.inf, -np.inf], np.nan)

# tipado de binarias
for col in binary_features:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").fillna(0).astype(int)

# tipado de categóricas
for col in categorical_features:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype("string")

# ordenar temporalmente
df_model = df_model.sort_values("merge_time").reset_index(drop=True)

# imputación manual de nulos estructurales
fill_zero_cols = [
    "delay_mean_linea",
    "delay_max_linea",
    "delay_std_linea",
    "paradas_retrasadas",
    "total_paradas",
    "pct_paradas_retrasadas",
    "delay_1_before_mean",
    "delay_2_before_mean",
    "delay_3_before_mean",
    "delay_acceleration_linea",
    "delay_x_aceleracion",
    "headway_mean_linea",
    "headway_std_linea",
    "headway_cv",
    "delay_rolling4_mean",
    "delay_rolling4_max",
    "headway_rolling4_std",
    "is_unscheduled",
    "num_updates",
    "match_key_nunique",
    "hour_sin",
    "hour_cos",
    "dow",
    "is_weekend",
    "seg_desde_ultima_alerta_linea",
    "colapso_linea",
]

for col in fill_zero_cols:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").fillna(0)

# recalcular derivadas por seguridad
if {"paradas_retrasadas", "total_paradas"}.issubset(df_model.columns):
    df_model["pct_paradas_retrasadas"] = (
        df_model["paradas_retrasadas"] / df_model["total_paradas"].clip(lower=1)
    )

if {"delay_mean_linea", "delay_1_before_mean"}.issubset(df_model.columns):
    df_model["delay_acceleration_linea"] = (
        df_model["delay_mean_linea"] - df_model["delay_1_before_mean"]
    )

if {"delay_mean_linea", "delay_acceleration_linea"}.issubset(df_model.columns):
    df_model["delay_x_aceleracion"] = (
        df_model["delay_mean_linea"] * df_model["delay_acceleration_linea"].clip(lower=0)
    )

if {"headway_std_linea", "headway_mean_linea"}.issubset(df_model.columns):
    df_model["headway_cv"] = (
        df_model["headway_std_linea"] / df_model["headway_mean_linea"].clip(lower=1)
    )

print("Shape tras limpieza:", df_model.shape)
print(df_model[features + [target]].isna().sum().sort_values(ascending=False))

Shape tras limpieza: (755443, 30)
delay_mean_linea                 0
delay_max_linea                  0
delay_std_linea                  0
paradas_retrasadas               0
total_paradas                    0
pct_paradas_retrasadas           0
delay_1_before_mean              0
delay_2_before_mean              0
delay_3_before_mean              0
delay_acceleration_linea         0
delay_x_aceleracion              0
headway_mean_linea               0
headway_std_linea                0
headway_cv                       0
delay_rolling4_mean              0
delay_rolling4_max               0
headway_rolling4_std             0
is_unscheduled                   0
num_updates                      0
match_key_nunique                0
hour_sin                         0
hour_cos                         0
dow                              0
is_weekend                       0
seg_desde_ultima_alerta_linea    0
colapso_linea                    0
direction                        0
route_id             

5) Comprobación de que todo esta bien antes de empezar a entrenar el modelo

In [139]:
# 1. Target check
counts = df_model[target].value_counts(normalize=True)
print(f"\nDistribución target:\n{counts}")

# 2. Nulos persistentes
nulos = df_model[features].isna().sum()
if nulos.sum() > 0:
    print(f"\n¡ADVERTENCIA! Nulos detectados:\n{nulos[nulos > 0]}")
else:
    print("\n✅ Sin nulos en las features.")

# 3. Tipos de datos
print(f"\nTipos de columnas:\n{df_model[features + [target]].dtypes.value_counts()}")




Distribución target:
alert_in_next_15m_max
0    0.856214
1    0.143786
Name: proportion, dtype: float64

✅ Sin nulos en las features.

Tipos de columnas:
float32    13
int64       7
float64     6
string      2
Int32       1
Name: count, dtype: int64


6) Mas comprobaciones (sobre todo de nulos, ya que no son compatibles):

In [140]:
X = df_model[features].copy()
y = df_model[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nNaN en X:")
print(X.isna().sum())

print("\nNaN en y:")
print(y.isna().sum())

X shape: (755443, 28)
y shape: (755443,)

NaN en X:
delay_mean_linea                 0
delay_max_linea                  0
delay_std_linea                  0
paradas_retrasadas               0
total_paradas                    0
pct_paradas_retrasadas           0
delay_1_before_mean              0
delay_2_before_mean              0
delay_3_before_mean              0
delay_acceleration_linea         0
delay_x_aceleracion              0
headway_mean_linea               0
headway_std_linea                0
headway_cv                       0
delay_rolling4_mean              0
delay_rolling4_max               0
headway_rolling4_std             0
is_unscheduled                   0
num_updates                      0
match_key_nunique                0
hour_sin                         0
hour_cos                         0
dow                              0
is_weekend                       0
seg_desde_ultima_alerta_linea    0
colapso_linea                    0
direction                        0
rou

8) Split/Train test

In [141]:
X = df_model[features].copy()
y = df_model[target].copy()

split = int(len(df_model) * 0.8)

X_train = X.iloc[:split].copy()
X_test  = X.iloc[split:].copy()

y_train = y.iloc[:split].copy()
y_test  = y.iloc[split:].copy()

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

print("Rango temporal train:",
      df_model.iloc[:split]["merge_time"].min(),
      "->",
      df_model.iloc[:split]["merge_time"].max())

print("Rango temporal test:",
      df_model.iloc[split:]["merge_time"].min(),
      "->",
      df_model.iloc[split:]["merge_time"].max())

Train: (604354, 28) (604354,)
Test: (151089, 28) (151089,)
Rango temporal train: 2025-01-01 00:00:00 -> 2025-10-26 09:30:00
Rango temporal test: 2025-10-26 09:30:00 -> 2025-12-31 23:30:00


9) Ultima comprobación de nulos

In [142]:
print("NaN en X_train:")
print(X_train.isna().sum().sort_values(ascending=False))

print("\nNaN en X_test:")
print(X_test.isna().sum().sort_values(ascending=False))

print("\nNaN en y_train:", y_train.isna().sum())
print("NaN en y_test:", y_test.isna().sum())

num_train = X_train.select_dtypes(include=[np.number])
num_test = X_test.select_dtypes(include=[np.number])

print("\nInf en X_train:")
print(np.isinf(num_train).sum())

print("\nInf en X_test:")
print(np.isinf(num_test).sum())

NaN en X_train:
delay_mean_linea                 0
delay_max_linea                  0
delay_std_linea                  0
paradas_retrasadas               0
total_paradas                    0
pct_paradas_retrasadas           0
delay_1_before_mean              0
delay_2_before_mean              0
delay_3_before_mean              0
delay_acceleration_linea         0
delay_x_aceleracion              0
headway_mean_linea               0
headway_std_linea                0
headway_cv                       0
delay_rolling4_mean              0
delay_rolling4_max               0
headway_rolling4_std             0
is_unscheduled                   0
num_updates                      0
match_key_nunique                0
hour_sin                         0
hour_cos                         0
dow                              0
is_weekend                       0
seg_desde_ultima_alerta_linea    0
colapso_linea                    0
direction                        0
route_id                         0
dtyp

10) Creación del pipeline

In [143]:
# 1. Asegúrate de que estas listas están actualizadas con las nuevas columnas
categorical_features = ["direction", "route_id"]

# Creamos la lista numérica excluyendo las categóricas automáticamente
numeric_features = [c for c in features if c not in categorical_features]

# 2. Definición de Transformadores
numeric_transformer = Pipeline(steps=[
    # Usamos 'median' porque los retrasos suelen tener valores atípicos (outliers)
    ("imputer", SimpleImputer(strategy="median")), 
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    # Si falta una categoría, usamos la más frecuente
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # handle_unknown='ignore' es vital por si aparece una ruta nueva en el test
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)) 
])

# 3. Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 4. Pipeline Final
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,           # Aumentado para asegurar convergencia
        class_weight="balanced", # Mantenlo: es clave para tu desequilibrio de clases
        random_state=42,
        solver="lbfgs"
    ))
])

# 5. Entrenamiento
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

11) Entrenamiento del modelo

In [144]:

y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

F1: 0.42815144766146995
Precision: 0.2838915470494418
Recall: 0.8704944756384713

Classification report:
              precision    recall  f1-score   support

           0       0.97      0.62      0.76    129005
           1       0.28      0.87      0.43     22084

    accuracy                           0.66    151089
   macro avg       0.62      0.75      0.59    151089
weighted avg       0.87      0.66      0.71    151089


Confusion matrix:
[[80513 48492]
 [ 2860 19224]]


12) Evaluacion con otros umbrales

In [145]:
for thr in [0.5, 0.3, 0.2, 0.1, 0.05]:
    y_pred_thr = (y_prob >= thr).astype(int)
    print(f"\nUmbral = {thr}")
    print(classification_report(y_test, y_pred_thr, zero_division=0))


Umbral = 0.5
              precision    recall  f1-score   support

           0       0.97      0.62      0.76    129005
           1       0.28      0.87      0.43     22084

    accuracy                           0.66    151089
   macro avg       0.62      0.75      0.59    151089
weighted avg       0.87      0.66      0.71    151089


Umbral = 0.3
              precision    recall  f1-score   support

           0       0.98      0.40      0.57    129005
           1       0.21      0.95      0.35     22084

    accuracy                           0.48    151089
   macro avg       0.60      0.68      0.46    151089
weighted avg       0.87      0.48      0.54    151089


Umbral = 0.2
              precision    recall  f1-score   support

           0       0.98      0.32      0.48    129005
           1       0.20      0.97      0.33     22084

    accuracy                           0.41    151089
   macro avg       0.59      0.64      0.40    151089
weighted avg       0.87      0.4

14) Probabilidades para comprobar

In [146]:
print("Probabilidades:")
print("min:", y_prob.min())
print("max:", y_prob.max())
print("mean:", y_prob.mean())

Probabilidades:
min: 1.892878417028106e-40
max: 0.9887301474540884
mean: 0.40525850058112484


In [147]:
import optuna

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

# split temporal interno sobre train
split_val = int(len(X_train) * 0.8)

X_tr = X_train.iloc[:split_val].copy()
y_tr = y_train.iloc[:split_val].copy()

X_val = X_train.iloc[split_val:].copy()
y_val = y_train.iloc[split_val:].copy()

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    threshold = trial.suggest_float("threshold", 0.1, 0.9)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    pipeline_trial = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=C,
            class_weight=class_weight,
            max_iter=300,
            random_state=42,
            solver="lbfgs"
        ))
    ])

    pipeline_trial.fit(X_tr, y_tr)

    y_val_prob = pipeline_trial.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_prob >= threshold).astype(int)

    return f1_score(y_val, y_val_pred, zero_division=0)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Mejor F1:", study.best_value)
print("Mejores parámetros:", study.best_params)

[I 2026-04-05 16:49:46,660] A new study created in memory with name: no-name-a8e60c83-30b8-40a1-bd7e-84106503fc2e
[I 2026-04-05 16:49:58,920] Trial 0 finished with value: 0.32210123008671104 and parameters: {'C': 0.04548178590705045, 'class_weight': 'balanced', 'threshold': 0.24059648422135888}. Best is trial 0 with value: 0.32210123008671104.
[I 2026-04-05 16:50:09,437] Trial 1 finished with value: 0.4460681684951009 and parameters: {'C': 0.0037456844224273847, 'class_weight': 'balanced', 'threshold': 0.608081546084304}. Best is trial 1 with value: 0.4460681684951009.
[I 2026-04-05 16:50:20,910] Trial 2 finished with value: 0.00011970313622216902 and parameters: {'C': 0.017931488577574966, 'class_weight': None, 'threshold': 0.8630243056198368}. Best is trial 1 with value: 0.4460681684951009.
[I 2026-04-05 16:50:31,606] Trial 3 finished with value: 0.38556617348632993 and parameters: {'C': 0.008915602758965501, 'class_weight': None, 'threshold': 0.30834372548400435}. Best is trial 1 wi

Mejor F1: 0.45815238014926
Mejores parámetros: {'C': 0.006208282961956851, 'class_weight': None, 'threshold': 0.22362580626708306}


In [148]:
best_params = study.best_params

best_C = best_params["C"]
best_class_weight = best_params["class_weight"]
best_threshold = best_params["threshold"]

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=best_C,
        class_weight=best_class_weight,
        max_iter=300,
        random_state=42,
        solver="lbfgs"
    ))
])

final_pipeline.fit(X_train, y_train)

y_test_prob = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("F1 test:", f1_score(y_test, y_test_pred, zero_division=0))
print("Precision test:", precision_score(y_test, y_test_pred, zero_division=0))
print("Recall test:", recall_score(y_test, y_test_pred, zero_division=0))

print("\nClassification report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

F1 test: 0.47230506690483404
Precision test: 0.3645517886599207
Recall test: 0.6704854193080964

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86    129005
           1       0.36      0.67      0.47     22084

    accuracy                           0.78    151089
   macro avg       0.65      0.74      0.67    151089
weighted avg       0.85      0.78      0.80    151089


Confusion matrix:
[[103195  25810]
 [  7277  14807]]


In [149]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import loguniform
from sklearn.metrics import f1_score
import numpy as np

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

# split temporal interno sobre train para luego elegir threshold en validación
split_val = int(len(X_train) * 0.8)

X_tr = X_train.iloc[:split_val].copy()
y_tr = y_train.iloc[:split_val].copy()

X_val = X_train.iloc[split_val:].copy()
y_val = y_train.iloc[split_val:].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline_rs = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=300,
        random_state=42,
        solver="lbfgs"
    ))
])

param_distributions = {
    "model__C": loguniform(1e-3, 10.0),
    "model__class_weight": [None, "balanced"]
}

tscv = TimeSeriesSplit(n_splits=3)

random_search = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=tscv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

random_search.fit(X_tr, y_tr)

print("Mejores hiperparámetros RandomizedSearchCV:")
print(random_search.best_params_)
print("Mejor F1 CV:", random_search.best_score_)

# búsqueda del mejor threshold en validación, igual que con Optuna
best_pipeline_rs = random_search.best_estimator_
y_val_prob_rs = best_pipeline_rs.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
best_threshold_rs = 0.5
best_f1_rs = -1

for thr in thresholds:
    y_val_pred_rs = (y_val_prob_rs >= thr).astype(int)
    f1 = f1_score(y_val, y_val_pred_rs, zero_division=0)
    if f1 > best_f1_rs:
        best_f1_rs = f1
        best_threshold_rs = thr

print("Mejor threshold en validación:", best_threshold_rs)
print("F1 validación con mejor threshold:", best_f1_rs)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Mejores hiperparámetros RandomizedSearchCV:
{'model__C': np.float64(0.679657809075816), 'model__class_weight': 'balanced'}
Mejor F1 CV: 0.4205220863174051
Mejor threshold en validación: 0.6500000000000001
F1 validación con mejor threshold: 0.469685304590511


In [150]:
best_C_rs = random_search.best_params_["model__C"]
best_class_weight_rs = random_search.best_params_["model__class_weight"]

categorical_features = ["direction", "route_id"]
numeric_features = [c for c in features if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

final_pipeline_rs = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=best_C_rs,
        class_weight=best_class_weight_rs,
        max_iter=300,
        random_state=42,
        solver="lbfgs"
    ))
])

final_pipeline_rs.fit(X_train, y_train)

y_test_prob_rs = final_pipeline_rs.predict_proba(X_test)[:, 1]
y_test_pred_rs = (y_test_prob_rs >= best_threshold_rs).astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix

print("F1 test:", f1_score(y_test, y_test_pred_rs, zero_division=0))
print("Precision test:", precision_score(y_test, y_test_pred_rs, zero_division=0))
print("Recall test:", recall_score(y_test, y_test_pred_rs, zero_division=0))

print("\nClassification report:\n")
print(classification_report(y_test, y_test_pred_rs, zero_division=0))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_test_pred_rs))

print("\nResumen RandomizedSearchCV")
print("C:", best_C_rs)
print("class_weight:", best_class_weight_rs)
print("threshold:", best_threshold_rs)

F1 test: 0.4743519831315665
Precision test: 0.39623443668387487
Recall test: 0.5908349936605687

Classification report:

              precision    recall  f1-score   support

           0       0.92      0.85      0.88    129005
           1       0.40      0.59      0.47     22084

    accuracy                           0.81    151089
   macro avg       0.66      0.72      0.68    151089
weighted avg       0.85      0.81      0.82    151089


Matriz de confusión:
[[109123  19882]
 [  9036  13048]]

Resumen RandomizedSearchCV
C: 0.679657809075816
class_weight: balanced
threshold: 0.6500000000000001
